# SDACS 튜토리얼

**군집드론 공역통제 자동화 시스템 (Swarm Drone Airspace Control System)**

이 노트북에서는 SDACS 시뮬레이션의 기본 사용법을 단계별로 안내합니다.

- 기본 시뮬레이션 실행
- KPI 결과 분석
- GPU 백엔드 확인
- 시나리오 실행

In [ ]:
# 셀 1: 환경 설정 및 import
import sys
import os

# 프로젝트 루트를 경로에 추가 (노트북이 notebooks/ 디렉토리에 있을 때)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
from simulation.simulator import SwarmSimulator
from simulation.apf_engine import get_apf_backend_info
from simulation.scenario_runner import list_scenarios, run_scenario

print(f"프로젝트 루트: {PROJECT_ROOT}")
print("모듈 로드 완료")

## 기본 시뮬레이션 실행

`SwarmSimulator`를 사용하여 드론 군집 시뮬레이션을 실행합니다.
- `seed`: 재현성을 위한 랜덤 시드
- `drones.default_count`: 시뮬레이션 드론 수
- `duration_s`: 시뮬레이션 시간 (초)

In [ ]:
# 셀 2: 기본 시뮬레이션 실행
import time

DRONE_COUNT = 30
DURATION_S = 60
SEED = 42

override = {"drones": {"default_count": DRONE_COUNT}}
sim = SwarmSimulator(seed=SEED, scenario_cfg=override)

t0 = time.monotonic()
result = sim.run(duration_s=DURATION_S)
elapsed = time.monotonic() - t0

print(f"시뮬레이션 완료: {DRONE_COUNT}대 드론, {DURATION_S}초, 실행시간 {elapsed:.2f}s")
print()
print(result.summary_table())

## KPI 결과 분석

시뮬레이션 결과에서 주요 성능 지표(KPI)를 추출하여 분석합니다.
- 이벤트 유형별 발생 횟수
- 비행 단계별 드론 분포
- 통신 버스 통계

In [ ]:
# 셀 3: 결과 분석 (KPI)

# 이벤트 타임라인 분석
events = sim.analytics.events
event_counts = {}
for ev in events:
    event_counts[ev["type"]] = event_counts.get(ev["type"], 0) + 1

print("=== 이벤트 유형별 발생 횟수 ===")
for etype, cnt in sorted(event_counts.items(), key=lambda x: -x[1]):
    print(f"  {etype:<30} {cnt:>6}")

# 비행 단계별 최종 분포
phase_counts = {}
for d in sim._drones.values():
    name = d.flight_phase.name
    phase_counts[name] = phase_counts.get(name, 0) + 1

print()
print("=== 비행 단계별 드론 분포 ===")
for phase, cnt in sorted(phase_counts.items(), key=lambda x: -x[1]):
    bar = chr(9608) * min(cnt, 30)
    print(f"  {phase:<20} {cnt:>4}  {bar}")

# 통신 버스 통계
cs = sim.comm_bus.stats
print()
print(f"=== 통신 통계 ===")
print(f"  전송: {cs['sent']}  수신: {cs['delivered']}  유실: {cs['dropped']}")

## GPU 백엔드 확인

APF(Artificial Potential Field) 충돌 회피 연산의 백엔드 정보를 확인합니다.
GPU가 감지되면 자동으로 CUDA 가속이 활성화됩니다.

In [ ]:
# 셀 4: GPU 백엔드 확인
backend = get_apf_backend_info()

print("=== APF 백엔드 정보 ===")
for key, value in backend.items():
    print(f"  {key}: {value}")

# PyTorch CUDA 확인
try:
    import torch
    print()
    print(f"PyTorch 버전: {torch.__version__}")
    print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
        print(f"CUDA 버전: {torch.version.cuda}")
except ImportError:
    print("\nPyTorch가 설치되지 않았습니다.")

## 시나리오 실행

SDACS는 8개의 사전 정의된 시나리오를 제공합니다.
여기서는 `high_density` (고밀도 교통) 시나리오를 실행합니다.

In [ ]:
# 셀 5: 시나리오 목록 확인 및 실행

# 시나리오 목록
scenarios = list_scenarios()
print("=== 사용 가능한 시나리오 ===")
for s in scenarios:
    if isinstance(s, dict):
        print(f"  {s.get('name', s):<28} {s.get('description', '')[:50]}")
    else:
        print(f"  {s}")

# high_density 시나리오 1회 실행
print()
print("=== high_density 시나리오 실행 ===")
results = run_scenario("high_density", n_runs=1, seed=42)

for k, v in results[0].items():
    if k not in ("run_idx", "config_params"):
        print(f"  {k}: {v}")